### 🖼️ Dataset Thumbnails (CohereLabs___aya_evaluation_suite)

![Thumbnail](../thumbnails/CohereLabs___aya_evaluation_suite_01.png)

In [ ]:
import random
from datasets import load_dataset, get_dataset_config_names
from collections import Counter
import time

# =============================================================================
# 💡 데이터셋 소개: Aya Evaluation Suite
# -----------------------------------------------------------------------------
# 데이터셋명: CohereLabs/aya_evaluation_suite
# 한글 제목: 아야 평가 스위트 (Aya Evaluation Suite)
# 의미 및 설명: 이 데이터셋은 전 세계 수많은 언어(약 100개 이상)의 다양한 문장 구조와 번역 능력을 평가하기 위해 크라우드소싱, 전문가 작성, 기계 생성 등으로 다채롭게 구축된 거대한 다국어 텍스트 데이터셋입니다.
# 🎯 실습 목표: 이 데이터셋을 이용해 '언어 패턴 탐지기'를 만들어, 특정 언어에서 질문과 답변의 쌍을 찾아내는 초급 AI의 논리적 사고 과정을 흉내 내봅니다.
# =============================================================================

# 상수로 정의합니다.
DATASET_NAME = "CohereLabs/aya_evaluation_suite"
SAMPLE_COUNT = 50 # 메모리 절약 및 빠른 실습을 위해 상위 50개 샘플만 사용합니다.

# -----------------------------------------------------------------------------
# 📌 1단계: 데이터셋 로딩 (스트리밍 우선 전략)
# -----------------------------------------------------------------------------
dataset = None
print("🌟 [튜터 코멘트] 데이터셋을 로드할 준비를 합니다. 스트리밍(streaming=True)으로 먼저 시도해볼게요!")

# 1. 사용 가능한 Config 확인
try:
    # 이 코드를 통해 사용 가능한 설정을 먼저 확인합니다.
    configs = get_dataset_config_names(DATASET_NAME)
    print(f"✅ 사용 가능한 Config 목록: {configs}")
    
    # 첫 번째 Config를 선택합니다.
    selected_config = configs[0]
except Exception as e:
    print(f"ℹ️ Config 확인 중 오류 발생: {e}. 기본 설정을 사용하겠습니다.")
    selected_config = None

# 2. 스트리밍 로드 시도 (가장 빠르고 효율적인 방법)
try:
    # streaming=True로 로드합니다. 메모리 효율성이 가장 좋습니다.
    dataset = load_dataset(DATASET_NAME, config_name=selected_config, split='test', streaming=True)
    print("✅ [성공] 스트리밍 모드(streaming=True)로 데이터셋을 성공적으로 로드했습니다! (메모리 절약 최고!)")
except Exception as e:
    print(f"\n⚠️ [경고] 스트리밍 로드 실패! (오류: {e.__class__.__name__}). 일반 로드로 전환합니다.")
    # 3. 스트리밍 실패 시: 일반 로드 (streaming=False)
    try:
        dataset = load_dataset(DATASET_NAME, config_name=selected_config, split='test')
        print("✅ [성공] 일반 로드 모드(streaming=False)로 데이터셋을 로드했습니다.")
    except Exception as inner_e:
        print(f"❌ [실패] 데이터셋 로드에 최종적으로 실패했습니다: {inner_e}")
        exit()


# -----------------------------------------------------------------------------
# 🚀 2단계: 샘플 데이터 준비 및 언어 통계 분석
# -----------------------------------------------------------------------------
print("\n" + "="*80)
print("✨ [미션 1] 언어 탐정 놀이: 데이터가 어떤 언어로 채워져 있는지 분석해 봅시다.")
print("="*80)

# 데이터를 직접 리스트로 변환하는 과정 (Constraint 준수)
# streaming이냐 아니냐에 따라 패턴이 다르므로, list(dataset.take(N)) 패턴 사용!
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋이거나 스트리밍처럼 처리해야 함
    sample_dataset_iterator = dataset.take(SAMPLE_COUNT)
    sample_data_list = list(sample_dataset_iterator)
else:
    # 일반 Dataset 객체인 경우
    sample_data_list = list(dataset.select(range(SAMPLE_COUNT)))

print(f"📚 분석 대상: 총 {len(sample_data_list)}개의 샘플을 분석할 것입니다.")

# 1. 언어별 빈도수 계산 (정량적 분석)
language_list = [sample['language'] for sample in sample_data_list if 'language' in sample]
language_counts = Counter(language_list)

print("\n--- 🌍 언어 분포 분석 ---")
# 상위 5개 언어만 보기 쉽게 출력합니다.
print("✨ 데이터셋은 얼마나 다양한 언어로 구성되어 있을까요?")
for lang, count in language_counts.most_common(5):
    print(f"  - {lang.upper()} ([{lang}]): 약 {count}개 샘플")
print(f"💡 튜터 코멘트: 이 데이터를 보면 정말 글로벌한 거대 데이터셋이라는 걸 알 수 있죠? 데이터가 아주 다양해요!")


# -----------------------------------------------------------------------------
# 🧠 3단계: AI 논리 실습 - 질문/답변 쌍 탐지기 만들기
# -----------------------------------------------------------------------------
print("\n" + "="*80)
print("🤖 [미션 2] AI 시뮬레이션: '질문'과 '답변'의 패턴 찾기")
print("="*80)

print("💡 목표: 이 AI는 입력(inputs)이 질문 형태일 때, 출력(targets)이 그 질문에 대한 적절한 답변일 가능성이 높은지 패턴을 찾아봅니다.")

# 질문/답변 패턴을 추적할 변수
detected_pairs_count = 0
target_threshold_words = 3 # 답변이 최소 3단어 이상이어야 유효한 답변으로 간주

print(f"\n🔍 상위 {SAMPLE_COUNT}개 샘플 중 '질문-답변' 패턴을 탐지합니다...")

for i, sample in enumerate(sample_data_list):
    # 필수 Feature 체크
    if 'inputs' not in sample or 'targets' not in sample:
        continue
    
    inputs_text = sample['inputs'].strip()
    targets_text = sample['targets'].strip()

    # A. 질문 패턴 인식 (Heuristic Rule: ? 포함 또는 특정 단어 시작)
    is_question = (
        '?' in inputs_text or 
        (len(inputs_text) > 10 and inputs_text.lower().startswith(("what", "how", "why", "when", "where")))
    )
    
    # B. 답변 유효성 검사
    is_valid_answer = (
        targets_text and 
        len(targets_text.split()) >= target_threshold_words
    )

    # C. 패턴 매칭 및 결과 출력
    if is_question and is_valid_answer:
        # 성공적으로 패턴을 포착했을 때, 마치 AI가 정답을 찾은 것처럼 출력합니다.
        if detected_pairs_count < 5: # 과도한 출력을 막기 위해 상위 5개만 보여줍니다.
            print(f"\n✅ [{i+1}번째 샘플] [⭐ 발견! 질문-답변 패턴 감지] (Language: {sample.get('language', 'N/A')})")
            print(f"   ➡️ 질문 (Input): '{inputs_text[:70]}...'")
            print(f"   📚 답변 (Target): '{targets_text[:70]}...'")
            detected_pairs_count += 1
    
    # 학습자가 AI가 실패한 케이스도 이해할 수 있게 해주면 좋아요.
    elif not is_question and targets_text:
         pass # 너무 많은 출력을 방지하기 위해 건너뜁니다.

if detected_pairs_count == 0:
    print("\n😭 패턴을 찾지 못했어요! 데이터의 분포가 매우 다양해서, 우리가 설정한 단순 규칙으로는 알기 어려울 수 있어요. 더 정교한 LLM이 필요하겠네요!")
else:
    print("\n🎉 튜터 코멘트: 축하합니다! 우리는 간단한 '규칙 기반 탐지기'를 성공적으로 만들어 보았습니다. 이 원리가 바로 AI가 정보를 해석하고 패턴을 찾는 기본 논리입니다!")

print("\n" + "="*80)
print("🚀 실습 완료! 데이터를 구조화하고 패턴을 분석하는 과정이 AI 개발의 첫걸음이랍니다. 정말 잘 하셨어요! 👍")
print("="*80)